# <span style="color:#9947dc">HI-Small — Modelling (transaction scoring)</span>

**What this notebook is for.** The capstone is an agent system that investigates
money-laundering alerts. The model trained here is one of the agent's tools: it scores
every transfer, and each day the highest scores become the alerts the agent starts from.
So the questions are not "what is the accuracy" but:

1. If we can only alert a fixed small share of each day's transfers, how many laundering transfers do we catch, and how many alerts are wasted?
2. Of the laundering **schemes**, how many get at least one alert? One alert is all the agent needs to start rebuilding the case.
3. How much of the result comes from generator artefacts (payment format, the conversion step) rather than behaviour a real bank would see?

**Inputs:** `data/processed/features/` from `03_features.py` (point-in-time features built from the
full joined file). **Outputs:** `models/txn_scorer.joblib` (model + alert rule for the agent),
`models/test_scores.parquet`, `models/metrics.json`.

**Discipline.** Everything is fitted on **train** (1–6 Sept). Every choice (weighting, count
scaling, LightGBM settings, model, alert rule) is made on **validation** (7 Sept). **Test**
(8–10 Sept) is scored once, in section 9. Every metric uses `sample_weight` (all 1 on the full
file; inverse inclusion probabilities if you point this at the 02 sample).

**What changed after the first full run** (5 Nov). The first run gave test PR-AUC 0.611, but it had four problems:
- **Fixed threshold.** The score cutoff set on 7 Sept alerted 0.99% of test transfers instead of 0.5%. Alerts are now the **top X% of each day**, so the alert volume is fixed.
- **Moved-scheme rows.** Test contained 709 earlier rows of schemes that end in test. They are all positives with no negatives from their own days, which pushed precision from 8.3% to 13.0%. Test metrics now use only rows dated 8–10 Sept (the "test window"); moved rows are reported separately.
- **Drift in two counts.** Two 48 h counts drifted on validation (PSI ≈ 0.30). Section 3 tests whether total network volume explains it. Counts can now be rescaled by volume, and the scaling is chosen on validation.
- **Scheme catch.** Scheme catch is now shown for every feature set, not only the chosen one.

| # | Section |
|---|---|
| 0 | Setup and build check |
| 1 | The data the model sees |
| 2 | Leakage guard |
| 3 | Drift between splits — and is it just network volume? |
| 4 | Baseline: single features |
| 5 | LightGBM: weighting, count scaling, settings, feature sets |
| 6 | Logistic regression |
| 7 | Model choice on validation |
| 8 | Alert rule |
| 9 | Test results (scored once) |
| 10 | Who is caught and who is missed |
| 11 | Scheme-level catch rate (what the agent needs) |
| 12 | Why a transfer is flagged |
| 13 | Save for the agent layer |
| 14 | Findings |

## <span style="color:#9947dc">0. Setup</span>

**Why.** One place for every setting. If the feature files are missing (or were built before
the network-volume columns were added), the build scripts (`03_features.py`, then the
independent check `03b_verify_features.py`) are run from here. The notebook stops if any check fails.

Settings you may want to change:
- `TRAIN_NEG_FRACTION` — keep a fraction of training negatives (weights corrected) for a fast first pass, e.g. 0.25.
- `MAIN_BUDGET` — share of each day's transfers we can afford to alert. 0.5% is ~5× the true laundering rate.
- `NORMALISE` — `"auto"` (choose on validation), `True` or `False`: rescale window counts by network volume.
- `SPW_OPTIONS` — class weightings to compare. `"full"` was dropped after the first run (val PR-AUC 0.012 vs 0.433 unweighted, and 4 min to fit).
- `TUNE` — small LightGBM settings grid on validation (6 fits).
- `PRIMARY_SET` — feature set of the model handed to the agent (`all`, `no_format`, `no_generator`).

In [1]:
import json, subprocess, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

ROOT = Path.cwd() if (Path.cwd() / "03_features.py").exists() else Path.cwd().parent
sys.path.insert(0, str(ROOT / "src"))
import features as F
import modeling as M
import evaluation as E

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.width", 160, "display.max_columns", 30, "display.float_format", "{:,.6g}".format)

# ---------------------------------------------------------------- settings
FEATURES_DIR = ROOT / "data" / "processed" / "features"
MODELS_DIR = ROOT / "models"
BUILD_IF_MISSING = True        # run 03_features.py + 03b_verify_features.py if needed
BURN_IN = True                 # drop rows whose 48 h history window is not full from FITTING
TRAIN_NEG_FRACTION = 1.0       # e.g. 0.25 for a fast first pass (negatives re-weighted)
SEED = 42
BUDGETS = (0.001, 0.0025, 0.005, 0.01, 0.02)
MAIN_BUDGET = 0.005            # alert the top 0.5% of each day's transfers
NORMALISE = "auto"             # rescale window counts by network volume: "auto" | True | False
SPW_OPTIONS = (1.0, "sqrt")    # class weightings compared on validation
TUNE = True                    # small LightGBM grid on validation
TUNE_GRID = [dict(num_leaves=nl, min_child_samples=mc) for nl in (15, 31, 63) for mc in (20, 100)]
PRIMARY_SET = "all"            # model handed to the agent
BOOTSTRAP_B = 100              # resamples of sender accounts for confidence intervals

PURPLE, DARK, GREY = "#9947dc", "#6b21a8", "#9ca3af"
SET_COLOURS = {"all": PURPLE, "no_format": "#0ea5e9", "no_generator": "#f59e0b"}
print("project folder:", ROOT)
print("feature folder:", FEATURES_DIR)

project folder: c:\Users\almad\Documents\3rd year\1st semester\ProjFinalCurso\Project\Capstone-Project\alerts_model
feature folder: c:\Users\almad\Documents\3rd year\1st semester\ProjFinalCurso\Project\Capstone-Project\alerts_model\data\processed\features


In [2]:
def run(script, *args):
    # stream a pipeline script's output; stop on failure or on 'X / Y checks passed' with X != Y
    print(f"$ python {script} {' '.join(args)}")
    p = subprocess.Popen([sys.executable, str(ROOT / script), *args], cwd=ROOT,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    lines = []
    for line in p.stdout:
        print(line, end=""); lines.append(line)
    if p.wait() != 0:
        raise RuntimeError(f"{script} failed (exit {p.returncode})")
    for line in lines:
        if "checks passed" in line:
            a, b = line.split("checks passed")[0].split("/")
            if int(a) != int(b):
                raise RuntimeError(f"{script}: {line.strip()}")

needed = [FEATURES_DIR / f"{s}_{k}.parquet" for s in M.SPLITS for k in ("features", "meta")]
up_to_date = all(p.exists() for p in needed) and \
    set(F.META_COLUMNS) <= set(pd.read_parquet(FEATURES_DIR / "val_meta.parquet").columns)
if up_to_date:
    print("feature files found and up to date - nothing to build")
elif BUILD_IF_MISSING:
    print("feature files missing or built by an older 03_features.py - rebuilding")
    t0 = time.time()
    run("03_features.py")
    run("03b_verify_features.py")
    print(f"built in {time.time() - t0:.0f} s")
else:
    raise FileNotFoundError("feature files missing or out of date - run 03_features.py")

report = json.loads((FEATURES_DIR / "build_report.json").read_text())
print({k: report[k] for k in ("scored_rows", "scored_positives", "lookback_hours", "burn_in_rows", "n_hubs")})
print("build checks:", sum(report["checks"].values()), "/", len(report["checks"]), "passed")

feature files missing or built by an older 03_features.py - rebuilding
$ python 03_features.py 
[   0.3s] reading hi_small_joined.parquet (rows before 2022-09-11)...
[  46.2s] base: 5,077,228 rows kept, 1,108 residue rows dropped, 590,819 self-transfers, 4,522 positives
[  46.9s] FX table: 15 currencies, max spread within a pair 0.9409%
[  50.3s] hubs (>= 500 counterparties, train dates): 15
[  92.8s] cumulative event tables built
[ 409.7s] 16 point-in-time lookups done
Traceback (most recent call last):
  File "c:\Users\almad\Documents\3rd year\1st semester\ProjFinalCurso\Project\Capstone-Project\alerts_model\03_features.py", line 64, in <module>
    F.build_features(con, log=log)
    ~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^
  File "C:\Users\almad\Documents\3rd year\1st semester\ProjFinalCurso\Project\Capstone-Project\alerts_model\src\features.py", line 368, in build_features
    con.execute(f"""
    ~~~~~~~~~~~^^^^^
        CREATE OR REPLACE TABLE feat AS
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

RuntimeError: 03_features.py failed (exit 1)

## <span style="color:#9947dc">1. The data the model sees</span>

**Why.** Before fitting anything: how many rows and positives each split has, and what the
true (weighted) laundering rate is. PR-AUC only means something next to the prevalence of the
split it was measured on.

**How.** Load the three splits. Rows of a scheme that crosses a split boundary were moved to the
split of the scheme's last row (`moved_rows`), so test holds earlier rows of long schemes.
Those rows are kept for the scheme view (§11) but left out of the row-level test metrics, because
their own days' negatives are in train and validation. Self-transfers are not scored. The burn-in
rows (first 48 h) are not used to fit.

In [ ]:
splits = M.load_splits(FEATURES_DIR)
rows = []
for name, s in splits.items():
    m = s.meta
    rows.append({
        "split": name, "days": f"{m.Timestamp.min():%d %b} - {m.Timestamp.max():%d %b}",
        "rows": len(s), "positives": int(s.y.sum()),
        "weighted_rate": E.prevalence(s.y, s.w),
        "explained_pos": int((m.explained & (s.y == 1)).sum()),
        "unexplained_pos": int((~m.explained & (s.y == 1)).sum()),
        "moved_rows": int(m.scheme_moved.sum()),
        "burn_in_rows": int((~m.history_complete).sum()),
        "attempts": m["Attempt ID"].nunique(),
    })
overview = pd.DataFrame(rows).set_index("split")
display(overview)

train_fit = M.fit_rows(splits["train"], burn_in=BURN_IN)
if TRAIN_NEG_FRACTION < 1:
    train_fit = M.downsample_negatives(train_fit, TRAIN_NEG_FRACTION, seed=SEED)
val, test = splits["val"], splits["test"]
test_win = M.own_date(test)          # rows dated 8-10 Sept: the row-level test set
print(f"rows used to fit: {len(train_fit):,} ({int(train_fit.y.sum()):,} positives) "
      f"of {len(splits['train']):,} train rows; weighted rate {E.prevalence(train_fit.y, train_fit.w):.4%}")
print(f"test window (own date 8-10 Sept): {len(test_win):,} rows, {int(test_win.y.sum()):,} positives, "
      f"rate {E.prevalence(test_win.y, test_win.w):.4%}; moved rows left for §11: {len(test) - len(test_win):,}")

## <span style="color:#9947dc">2. Leakage guard</span>

**Why.** The EDA proved several columns predict the label perfectly because they come from the
patterns file or the sampling (`Attempt ID`, `Pattern Type`, `scheme_truncated`, `weight`, seeds…),
and IDs let a model memorise accounts. The network-volume columns (`net_*`) are meta too: they are
only used to rescale counts. None of these may reach a model.

In [ ]:
for name, s in splits.items():
    M.assert_no_leak(s.X)
    assert not set(s.X.columns) & {"net_48h", "net_24h", "net_60m"}
print("no leak / ID / network-volume columns in X  ✓\n")
groups = {"counts (48 h / 24 h / 60 min windows)": F.COUNT_FEATURES, "flags": F.FLAG_FEATURES,
          "amount": F.AMOUNT_FEATURES, "categorical": F.CATEGORICAL_FEATURES}
for g, cols in groups.items():
    print(f"{g:<40} {len(cols):>2}  {', '.join(cols)}")
print("\nfeature sets:", {k: len(v) for k, v in M.FEATURE_SETS.items()})
print("dropped from 'no_format':   ", F.FORMAT_FEATURES)
print("dropped from 'no_generator':", F.GENERATOR_FEATURES)

## <span style="color:#9947dc">3. Drift between splits — and is it just network volume?</span>

**Why.** A time split only tests generalisation if the features mean the same thing in every
split. In the first run two 48 h counts drifted between train and validation (`rcv_recv_48h` and
`pair_prev_48h`, PSI ≈ 0.30). The suspected cause is that the whole network is busier on some days
(test days ranged from 204k to 641k transfers): an account's count goes up with the network's
volume even if the account behaves the same.

**How.**
1. Per day: total network transfers in the 48 h window (`net_48h`) next to the mean of each count.
   If the daily count means move together with `net_48h`, volume is the cause.
2. PSI (population stability index; < 0.1 stable, 0.1–0.25 moderate, > 0.25 big) of every feature,
   train vs validation and test, with **raw** counts and with counts **rescaled by volume**.
   Rescaled = count × (train median network volume ÷ this row's network volume), rounded: "8
   transfers while the network did 20% more than usual" becomes 7. Counts stay whole numbers and a
   non-zero count never becomes 0, because otherwise "1 earlier transfer" on a busy day would turn
   into 0.73 and a tree split learned at 0.9 on train would read it as "no history". The reference
   volume comes from train only.

Whether the model uses rescaled counts is decided by validation PR-AUC in §5, not here.

In [ ]:
NET_REF = M.network_reference(train_fit)
numeric = F.COUNT_FEATURES + F.FLAG_FEATURES + F.AMOUNT_FEATURES
others = lambda a, b: {"val": (a.X, a.w), "test": (b.X, b.w)}

drift_raw = E.psi_table(train_fit.X, train_fit.w, others(val, test_win), numeric, F.CATEGORICAL_FEATURES)
tr_n, va_n, te_n = (M.normalise_counts(s, NET_REF) for s in (train_fit, val, test_win))
drift_norm = E.psi_table(tr_n.X, tr_n.w, others(va_n, te_n), numeric, F.CATEGORICAL_FEATURES)

drift = drift_raw[["val", "test", "flag"]].join(
    drift_norm[["val", "test", "flag"]], lsuffix="_raw", rsuffix="_rescaled")
display(drift.loc[[c for c in drift.index if c in F.COUNT_FEATURES]])
print("raw:      ", drift_raw["flag"].value_counts().to_dict())
print("rescaled: ", drift_norm["flag"].value_counts().to_dict())

In [ ]:
# per-day view: does the count level follow network volume?
allrows = pd.concat([s.meta.assign(**s.X[F.COUNT_FEATURES]) for s in splits.values()], ignore_index=True)
allrows = allrows[~allrows.scheme_moved]                       # each row on its own date
allrows["day"] = allrows["Timestamp"].dt.date
top = drift_raw.loc[[c for c in drift_raw.index if c in F.COUNT_FEATURES]].index[:4].tolist()
daily = allrows.groupby("day").agg(split=("split", "first"), rows=("row_id", "size"),
                                   net_48h=("net_48h", "mean"), **{c: (c, "mean") for c in top})
for c in top:
    daily[c + "_rescaled"] = (allrows[c] * NET_REF[F.NET_FOR_COUNT[c]]
                              / allrows[F.NET_FOR_COUNT[c]].clip(lower=1)).groupby(allrows["day"]).mean()
daily["weekday"] = pd.to_datetime(daily.index).strftime("%a")
display(daily)
full = daily[daily.index >= min(train_fit.meta.Timestamp).date()]
print("correlation of the daily mean with daily network volume (days with full history):")
print({c: round(full[c].corr(full["net_48h"]), 3) for c in top})

fig, axes = plt.subplots(1, 2, figsize=(13, 3.8))
axes[0].bar(daily.index.astype(str), daily["net_48h"], color=[SET_COLOURS["all"] if s == "train" else
            SET_COLOURS["no_format"] if s == "val" else SET_COLOURS["no_generator"] for s in daily["split"]])
axes[0].set_title("Network transfers in the 48 h window (mean per day)"); axes[0].tick_params(axis="x", rotation=45)
ref = daily.loc[full.index[full.split == "train"]]
for c, col in zip(top, [PURPLE, "#0ea5e9", "#f59e0b", "#10b981"]):
    axes[1].plot(daily.index.astype(str), daily[c] / ref[c].mean(), color=col, label=c)
    axes[1].plot(daily.index.astype(str), daily[c + "_rescaled"] / ref[c + "_rescaled"].mean(), color=col, ls="--")
axes[1].axhline(1, color=GREY, lw=1); axes[1].tick_params(axis="x", rotation=45)
axes[1].set_title("Daily mean ÷ train mean (solid raw, dashed rescaled)"); axes[1].legend(fontsize=8)
plt.tight_layout(); plt.show()

## <span style="color:#9947dc">4. Baseline: single features</span>

**Why.** A model is only worth having if it beats the best single column.

**How.** Each numeric feature is used alone as a score. Whether "higher" or "lower" is riskier is
decided on train; PR-AUC is then measured on validation. Lift = PR-AUC ÷ prevalence. The best one
becomes the `rule` model. Raw counts.

In [ ]:
single = M.single_feature_table(train_fit, val)
display(single)
rule = M.fit_rule(train_fit, val)
print(f"rule model: score = {'+' if rule.sign > 0 else '-'}{rule.feature}")

## <span style="color:#9947dc">5. LightGBM</span>

**Why.** Gradient-boosted trees handle non-monotonic effects (amount), interactions (e.g. new
pair × conversion) and categories natively. The best model in the first run by a wide margin.

**How.** Four steps, each decided by validation PR-AUC, each building on the previous one:
1. **Class weighting.** Unweighted won the first run (0.433 vs 0.087 square-root); checked again.
2. **Count scaling.** Raw counts vs counts rescaled by network volume (§3).
3. **Settings.** A small grid over leaf count and minimum leaf size. Validation has only a few
   hundred positives, so a difference of a few hundredths is noise; the grid is kept small for that reason.
4. **Feature sets.** The chosen settings are refitted without payment format, and without payment
   format and the conversion step.

In [ ]:
def fit_eval(train, valid, feats, **kw):
    t0 = time.time()
    mdl = M.fit_lgbm(train, valid, feats, seed=SEED, **kw)
    ap = E.headline(valid.y, mdl.predict_score(valid.X), valid.w)["pr_auc"]
    return mdl, ap, time.time() - t0

steps = []
# 1. weighting (raw counts)
best = None
for spw in SPW_OPTIONS:
    mdl, ap, sec = fit_eval(train_fit, val, M.FEATURE_SETS["all"], spw=spw)
    steps.append({"step": "1 weighting", "option": f"scale_pos_weight={spw}", "best_iteration": mdl.best_iteration,
                  "val_pr_auc": ap, "seconds": sec})
    if best is None or ap > best[1]:
        best = (spw, ap, mdl)
BEST_SPW = best[0]

# 2. count scaling
if NORMALISE == "auto":
    mdl, ap, sec = fit_eval(tr_n, va_n, M.FEATURE_SETS["all"], spw=BEST_SPW)
    steps.append({"step": "2 count scaling", "option": "rescaled by network volume",
                  "best_iteration": mdl.best_iteration, "val_pr_auc": ap, "seconds": sec})
    USE_NORM = ap > best[1]
    if USE_NORM:
        best = (BEST_SPW, ap, mdl)
else:
    USE_NORM = bool(NORMALISE)
    if USE_NORM:
        mdl, ap, sec = fit_eval(tr_n, va_n, M.FEATURE_SETS["all"], spw=BEST_SPW)
        best = (BEST_SPW, ap, mdl)

# from here on every model uses the chosen representation
rep = (lambda s: M.normalise_counts(s, NET_REF)) if USE_NORM else (lambda s: s)
TR, VA, TE, TE_ALL = rep(train_fit), rep(val), rep(test_win), rep(test)

# 3. settings
BEST_PARAMS, best_ap, lgbm_all = {}, best[1], best[2]
if TUNE:
    for p in TUNE_GRID:
        mdl, ap, sec = fit_eval(TR, VA, M.FEATURE_SETS["all"], spw=BEST_SPW, params=p)
        steps.append({"step": "3 settings", "option": str(p), "best_iteration": mdl.best_iteration,
                      "val_pr_auc": ap, "seconds": sec})
        if ap > best_ap:
            BEST_PARAMS, best_ap, lgbm_all = p, ap, mdl

steps = pd.DataFrame(steps)
display(steps)
print(f"chosen: scale_pos_weight={BEST_SPW}, counts {'rescaled' if USE_NORM else 'raw'}, "
      f"settings {BEST_PARAMS or 'default'}  ->  val PR-AUC {best_ap:.4f}")

In [ ]:
# 4. feature sets with the chosen settings
lgbm = {"all": lgbm_all}
for fs in ("no_format", "no_generator"):
    t0 = time.time()
    lgbm[fs] = M.fit_lgbm(TR, VA, M.FEATURE_SETS[fs], spw=BEST_SPW, params=BEST_PARAMS, seed=SEED)
    print(f"lgbm / {fs:<13} best iteration {lgbm[fs].best_iteration:>5}  ({time.time() - t0:.0f} s)")

fig, ax = plt.subplots(figsize=(8, 3.5))
for fs, mdl in lgbm.items():
    curve = mdl.history["val"]["average_precision"]
    ax.plot(curve, color=SET_COLOURS[fs], label=f"{fs} (best it {mdl.best_iteration})")
    ax.axvline(mdl.best_iteration, color=SET_COLOURS[fs], ls=":", lw=1)
ax.set_xlabel("boosting iteration"); ax.set_ylabel("validation PR-AUC")
ax.set_title("LightGBM: validation PR-AUC while training"); ax.legend(); ax.grid(alpha=.3)
plt.tight_layout(); plt.show()

## <span style="color:#9947dc">6. Logistic regression</span>

**Why.** A transparent model whose coefficients can be read in the report. It needs more
preparation than trees:
- counts → `log1p` then standardised (they are heavy-tailed);
- amount → 10 quantile buckets, because risk by amount goes up and then down again (EDA v2);
- categories → one-hot, levels with fewer than 100 train rows grouped as "infrequent";
- strongly correlated numeric pairs (Spearman > 0.85 on train) → one of each dropped;
- `class_weight="balanced"`.

Same count representation as LightGBM.

In [ ]:
drop, pairs = M.correlated_to_drop(TR.X, F.COUNT_FEATURES + F.FLAG_FEATURES)
print("most correlated numeric pairs on train:")
display(pairs.head(6))
print("dropped for logistic regression:", drop)

logreg = {}
for fs, cols in M.FEATURE_SETS.items():
    t0 = time.time()
    logreg[fs] = M.fit_logreg(TR, cols, drop=drop, seed=SEED)
    ap = E.headline(VA.y, logreg[fs].predict_score(VA.X), VA.w)["pr_auc"]
    print(f"logreg / {fs:<13} val PR-AUC {ap:.4f}   ({time.time() - t0:.0f} s)")
display(M.logreg_coefficients(logreg["all"]).head(12))

## <span style="color:#9947dc">7. Model choice on validation</span>

**Why.** Pick the model for the agent without looking at test.

**How.** Every model × feature set scored on validation. The model handed to the agent is the
best one **within `PRIMARY_SET`**; the other feature sets are reported next to it so the report
can say how much the generator artefacts are worth.

In [ ]:
models = {("rule", "all"): rule}
models.update({("logreg", fs): m for fs, m in logreg.items()})
models.update({("lgbm", fs): m for fs, m in lgbm.items()})
X_for = lambda k, s_rep, s_raw: s_raw.X if k[0] == "rule" else s_rep.X   # rule uses raw counts

val_scores = {k: m.predict_score(X_for(k, VA, val)) for k, m in models.items()}
val_table = pd.DataFrame([{"model": k[0], "features": k[1], **E.headline(val.y, s, val.w)}
                          for k, s in val_scores.items()]).sort_values("pr_auc", ascending=False)
display(val_table.reset_index(drop=True))

cand = val_table[val_table.features == PRIMARY_SET].iloc[0]
CHOSEN = (cand["model"], cand["features"])
chosen = models[CHOSEN]
print(f"chosen for the agent: {CHOSEN[0]} / {CHOSEN[1]}  (validation PR-AUC {cand['pr_auc']:.4f})")

## <span style="color:#9947dc">8. Alert rule</span>

**Why.** The score has to be turned into alerts. Two rules:
- **Daily top-k (used):** each day, alert the top `MAIN_BUDGET` share of that day's transfers. The
  alert volume is fixed by construction, which is what an investigation team (or the agent's
  compute budget) can plan for.
- **Fixed threshold (reference):** a score cutoff set on validation. Simpler, but the alert volume
  moves with the score distribution. In the first run, a 0.5% budget became 0.99% on test.

**How.** On validation (one day) the two rules are the same thing. The table shows what each
budget gives there; the fixed threshold is kept as a fallback in the saved bundle.

In [ ]:
s_val = val_scores[CHOSEN]
val_budget = E.daily_budget_table(val.y, s_val, val.w, val.meta.Timestamp.dt.date, BUDGETS)
display(val_budget[["budget", "alerts", "alerts_per_day", "precision", "recall", "lift"]])
THRESHOLD = E.threshold_for_budget(s_val, val.w, MAIN_BUDGET)
print(f"fallback fixed threshold for the {MAIN_BUDGET:.2%} budget: {THRESHOLD:.6g}")

## <span style="color:#9947dc">9. Test results (scored once)</span>

**Why.** The only numbers that go in the report as performance.

**How.**
- Row-level metrics use the **test window**: rows dated 8–10 Sept.
- PR-AUC is shown for every model next to the window's prevalence. The all-test-rows column is shown only for comparison with the first run.
- For the chosen model, precision and recall at each budget use daily top-k alerts. The fixed validation threshold is shown next to it to show its alert-volume drift.
- 95% intervals come from resampling sender **accounts**; rows of one account are not independent, so resampling rows would make the intervals too narrow.

**What to check.** The gap between `all` and `no_generator` is how much of the score comes from
the generator's mechanics.

In [ ]:
test_scores = {k: m.predict_score(X_for(k, TE, test_win)) for k, m in models.items()}
test_all_scores = {k: m.predict_score(X_for(k, TE_ALL, test)) for k, m in models.items()}
test_table = pd.DataFrame([{"model": k[0], "features": k[1], **E.headline(test_win.y, s, test_win.w),
                            "pr_auc_all_test_rows": E.headline(test.y, test_all_scores[k], test.w)["pr_auc"]}
                           for k, s in test_scores.items()]).sort_values("pr_auc", ascending=False)
display(test_table.reset_index(drop=True))

s_test = test_scores[CHOSEN]
day_win = test_win.meta.Timestamp.dt.date.to_numpy()
test_budget = E.daily_budget_table(test_win.y, s_test, test_win.w, day_win, BUDGETS)
fixed = E.budget_table(test_win.y, s_test, test_win.w, BUDGETS, thresholds_from=(s_val, val.w))
test_budget["fixed_threshold_alert_rate"] = fixed["alert_rate"].to_numpy()
test_budget["fixed_threshold_precision"] = fixed["precision"].to_numpy()
print(f"\n{CHOSEN[0]} / {CHOSEN[1]} on the test window, daily top-k alerts:")
display(test_budget[["budget", "alerts", "alerts_per_day", "precision", "recall", "lift",
                     "fixed_threshold_alert_rate", "fixed_threshold_precision"]])

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
E.plot_pr(axes[0], {f"{k[0]}/{k[1]}": s for k, s in test_scores.items() if k[1] == PRIMARY_SET},
          test_win.y, test_win.w, title=f"Test window PR curves - feature set '{PRIMARY_SET}'")
E.plot_pr(axes[1], {f"lgbm/{fs}": test_scores[("lgbm", fs)] for fs in lgbm},
          test_win.y, test_win.w, title="Test window PR curves - LightGBM by feature set")
main_recall = test_budget.loc[test_budget.budget == MAIN_BUDGET, "recall"].iloc[0]
for ax in axes:
    ax.axvline(main_recall, color=GREY, ls=":", lw=1)
plt.tight_layout(); plt.show()

In [ ]:
alerts = {fs: E.daily_topk(test_scores[("lgbm", fs)], day_win, test_win.w, MAIN_BUDGET) for fs in lgbm}
alert = E.daily_topk(s_test, day_win, test_win.w, MAIN_BUDGET)
groups = test_win.meta["acct_from"].to_numpy()
ci = []
for fs in lgbm:
    s = test_scores[("lgbm", fs)]
    ci.append({"model": f"lgbm/{fs}", **E.cluster_bootstrap(test_win.y, s, test_win.w, groups, "pr_auc", BOOTSTRAP_B, SEED)})
    for metric in ("recall", "precision"):
        ci.append({"model": f"lgbm/{fs}", **E.cluster_bootstrap(test_win.y, s, test_win.w, groups, metric,
                                                                 BOOTSTRAP_B, SEED, alert=alerts[fs])})
ci_table = pd.DataFrame(ci)
print(f"95% intervals, {BOOTSTRAP_B} resamples of sender accounts; recall / precision at the "
      f"{MAIN_BUDGET:.2%} daily budget")
display(ci_table)

## <span style="color:#9947dc">10. Who is caught and who is missed</span>

**Why.** One recall number hides very different groups. The first run caught 93.8% of explained
positives (scheme transfers) but 30.2% of unexplained ones, and none of the 196 non-ACH positives.
This section checks whether that is the model or the feature set: if the `no_format` model
catches non-ACH positives, payment format was teaching "non-ACH is safe".

**How.** Daily top-k alerts at the main budget on the test window, recall per group for each
LightGBM feature set side by side; then alerts, precision and recall per test day.

In [ ]:
def breakdown(col):
    t = None
    for fs in lgbm:
        b = E.recall_breakdown(test_win.meta, alerts[fs], col)
        b = b.rename(columns={"caught": f"caught_{fs}", "recall": f"recall_{fs}"})
        t = b if t is None else t.join(b.drop(columns="positives"))
    return t

for col in ("explained", "Payment Format", "Pattern Type"):
    print(f"recall by {col}, test window, {MAIN_BUDGET:.2%} daily budget:")
    display(breakdown(col))

per_day = E.per_day(day_win, test_win.y, alert)
print(f"per test day - {CHOSEN[0]}/{CHOSEN[1]}:")
display(per_day)

## <span style="color:#9947dc">11. Scheme-level catch rate (what the agent needs)</span>

**Why.** The agent does not need every laundering transfer to be alerted: one alert is enough
to start, and its tools (DuckDB over the full joined file) walk the money from there. So the
number that matters for the system is **how many schemes get at least one alert**, and how far
into the scheme the first alert comes.

**How.** All laundering attempts in test. Only rows dated 8–10 Sept can raise an alert (the system
goes live on 8 Sept); earlier rows of long schemes count as already-happened steps. A scheme is
caught if any of its in-window transfers is alerted. `first_alert_step` is the position of the
first alerted transfer in the whole scheme (0 = the scheme's very first transfer). Truncated
schemes (part of them after 10 Sept) are shown separately, since only complete schemes can be used
to score the agent's case reconstruction. Shown for every LightGBM feature set.

In [ ]:
in_window = ~test.meta["scheme_moved"].to_numpy(bool)
def scheme_alerts(fs):
    a = np.zeros(len(test), bool)
    a[in_window] = alerts[fs]                      # test_win rows are test rows in the same order
    return a

assert (test.meta.loc[in_window, "row_id"].to_numpy() == test_win.meta["row_id"].to_numpy()).all()
scheme_rows, schemes_by_set = [], {}
for fs in lgbm:
    sc = E.scheme_catch(test.meta, scheme_alerts(fs))
    schemes_by_set[fs] = sc
    for trunc, g in sc.groupby("truncated"):
        scheme_rows.append({"features": fs, "schemes": "truncated" if trunc else "complete", "n": len(g),
                            "caught": int(g.caught.sum()), "catch_rate": g.caught.mean(),
                            "row_recall": g.rows_alerted.sum() / g.rows.sum(),
                            "median_first_alert_step": g.loc[g.caught, "first_alert_step"].median()})
scheme_table = pd.DataFrame(scheme_rows)
display(scheme_table)

schemes = schemes_by_set[CHOSEN[1]] if CHOSEN[0] == "lgbm" else E.scheme_catch(test.meta, scheme_alerts("all"))
summary = E.scheme_summary(schemes)
print(f"by pattern - {CHOSEN[0]}/{CHOSEN[1]}:")
display(summary)

fig, ax = plt.subplots(figsize=(9, 3.8))
width = 0.27
pats = sorted(schemes_by_set["all"]["pattern"].unique())
for i, fs in enumerate(lgbm):
    sc = schemes_by_set[fs]
    rate = sc[~sc.truncated].groupby("pattern")["caught"].mean().reindex(pats)
    ax.barh(np.arange(len(pats)) + (i - 1) * width, rate, height=width, color=SET_COLOURS[fs], label=fs)
ax.set_yticks(np.arange(len(pats))); ax.set_yticklabels(pats); ax.set_xlim(0, 1.05)
ax.set_xlabel("share of complete schemes with at least one alert")
ax.legend(fontsize=8, loc="upper left", bbox_to_anchor=(1.01, 1))
ax.set_title(f"Complete test schemes caught at the {MAIN_BUDGET:.2%} daily budget"); plt.tight_layout(); plt.show()

## <span style="color:#9947dc">12. Why a transfer is flagged</span>

**Why.** The agent has to explain each alert to an investigator, and the report has to show the
model uses the signals the EDA found rather than something odd.

**How.** For LightGBM: share of total split gain per feature, and mean absolute SHAP value
(exact TreeSHAP, built into LightGBM) on a sample of test alerts. Then three example alerts with
their top reasons, as the agent tool returns them.

In [ ]:
if hasattr(chosen, "importance"):
    imp = chosen.importance()
    alerted_idx = np.flatnonzero(alert)
    rng = np.random.default_rng(SEED)
    pick = rng.choice(alerted_idx, min(2000, len(alerted_idx)), replace=False) if len(alerted_idx) else []
    contrib = chosen.contributions(TE.X.iloc[pick])
    shap_mean = contrib.drop(columns="bias").abs().mean().sort_values()

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    g = imp.set_index("feature")["gain_share"].sort_values()
    axes[0].barh(g.index, g.values, color=PURPLE); axes[0].set_title("Share of split gain")
    axes[1].barh(shap_mean.index, shap_mean.values, color=DARK)
    axes[1].set_title(f"Mean |SHAP| on {len(pick):,} test alerts (log-odds)")
    plt.tight_layout(); plt.show()
elif hasattr(chosen, "pipeline"):
    display(M.logreg_coefficients(chosen).head(20))
else:
    print(f"rule model: score = {'+' if chosen.sign > 0 else '-'}{chosen.feature} - nothing else to show")

In [ ]:
ex_idx = np.r_[np.flatnonzero(alert & (test_win.y == 1))[:2], np.flatnonzero(alert & (test_win.y == 0))[:1]]
examples = test_win.meta.iloc[ex_idx].assign(case=["true alert"] * min(2, (alert & (test_win.y == 1)).sum())
                                             + ["false alert"] * min(1, (alert & (test_win.y == 0)).sum()))
if hasattr(chosen, "contributions"):
    reasons = M.reason_text(chosen.contributions(TE.X.iloc[ex_idx]), TE.X.iloc[ex_idx][chosen.features])
    examples = examples.assign(score=s_test[ex_idx], reasons=reasons)
with pd.option_context("display.max_colwidth", 200):
    display(examples[[c for c in ["case", "Timestamp", "acct_from", "acct_to", "Payment Format", "amount_usd",
                                  "Pattern Type", "score", "reasons"] if c in examples]])

## <span style="color:#9947dc">13. Save for the agent layer</span>

**Why.** The agent's scoring tool loads one file and scores exactly as evaluated here.

**What is saved.**
- `models/txn_scorer.joblib`: the model; the alert budget (daily top-k share, the main rule); the fallback fixed threshold; the network-volume reference if counts are rescaled; and provenance. Load it with `modeling.load_bundle`.
- How to score: `modeling.score_transactions(bundle, X, meta)` takes rows of a `*_features` / `*_meta` pair. It returns the score, the daily top-k alert flag and the top reasons. Pass `meta`: it carries the timestamp for the daily budget and the network volume for rescaling.
- `models/test_scores.parquet`: `row_id`, score and alert for every test-window transfer, so the agent evaluation starts from the same alerts.
- `models/metrics.json`: the tables above, for the report.

In [ ]:
MODELS_DIR.mkdir(exist_ok=True)
info = {
    "features_dir": str(FEATURES_DIR), "chosen": list(CHOSEN), "scale_pos_weight": str(BEST_SPW),
    "counts_rescaled": bool(USE_NORM), "lgbm_params": BEST_PARAMS, "alert_rule": "daily_topk",
    "burn_in": BURN_IN, "train_neg_fraction": TRAIN_NEG_FRACTION, "seed": SEED,
    "feature_build": {k: report[k] for k in ("scored_rows", "lookback_hours", "split_days", "cutoff")},
    "val_pr_auc": float(val_table.query("model == @CHOSEN[0] and features == @CHOSEN[1]")["pr_auc"].iloc[0]),
    "test_window_pr_auc": float(test_table.query("model == @CHOSEN[0] and features == @CHOSEN[1]")["pr_auc"].iloc[0]),
}
M.save_bundle(MODELS_DIR / "txn_scorer.joblib", chosen, THRESHOLD, MAIN_BUDGET, info,
              net_ref=NET_REF if USE_NORM else None)

pd.DataFrame({"row_id": test_win.meta["row_id"], "score": s_test, "alert": alert}) \
  .to_parquet(MODELS_DIR / "test_scores.parquet", index=False)

def records(df):
    return json.loads(df.reset_index().to_json(orient="records", default_handler=str))
metrics = {"info": info, "overview": records(overview), "drift": records(drift), "daily_volume": records(daily),
           "single_feature": records(single), "lgbm_steps": records(steps),
           "validation": records(val_table), "test_window": records(test_table),
           "test_budget": records(test_budget), "ci": records(ci_table),
           "schemes": records(scheme_table), "scheme_summary": records(summary), "per_day": records(per_day)}
(MODELS_DIR / "metrics.json").write_text(json.dumps(metrics, indent=2, default=str))

# round trip: the saved bundle, given raw features + meta, reproduces the scores and alerts above
b = M.load_bundle(MODELS_DIR / "txn_scorer.joblib")
rt = M.score_transactions(b, test_win.X, test_win.meta, explain=False)
assert np.allclose(rt["score"].to_numpy(), s_test) and (rt["alert"].to_numpy() == alert).all()
print("round trip ok; saved:", *[p.name for p in sorted(MODELS_DIR.iterdir())])

## <span style="color:#9947dc">14. Findings</span>

*Fill in after running. Convention: every number written here must be visible in an output
above — copy, don't retype.*

| Question | Where | Result |
|---|---|---|
| Does network volume explain the count drift? (daily correlation; PSI raw vs rescaled) | §3 | |
| Best single feature and its lift | §4 | |
| Weighting / count scaling / settings chosen | §5 steps table | |
| Model for the agent (val PR-AUC) | §7 | |
| Test-window PR-AUC vs prevalence, with 95% CI — all / no_format / no_generator | §9 | |
| Precision / recall / alerts per day at the 0.5% daily budget | §9 | |
| Fixed threshold: realised alert rate on test | §9 | |
| Recall explained vs unexplained; non-ACH recall per feature set | §10 | |
| Complete schemes caught — all / no_format / no_generator | §11 | |
| Top features by gain / SHAP — consistent with EDA v2? | §12 | |

**Limitations to state in the report.**
- 10 days of synthetic data: a structural hold-out, not a drift test.
- Payment format and the conversion step are generator mechanics.
- Training uses 3–6 Sept only; 1–2 Sept is the 48 h burn-in.
- Validation is a single day with a few hundred positives, so small differences between settings are noise.
- The daily alert budget is our choice, not measured staffing.